# Train Custom SpaCy NER on MedMentions

This notebook trains a SpaCy EntityRecognizer from scratch on the [MedMentions](https://huggingface.co/datasets/Aremaki/MedMentions) dataset.

**Steps:**
1. Install dependencies
2. Load & preprocess MedMentions (127 UMLS types → 10 NER categories)
3. Convert to SpaCy DocBin format
4. Train with `spacy train` on GPU
5. Evaluate & download the model


## 1. Install Dependencies

In [1]:
!pip install -q spacy datasets seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## 2. Load MedMentions & Map UMLS Types

In [2]:
from collections import Counter
from datasets import load_dataset

# ── UMLS Semantic Type → High-Level NER Category ──────────────────────
UMLS_TYPE_TO_LABEL = {
    # DISEASE
    "T047": "DISEASE", "T019": "DISEASE", "T046": "DISEASE", "T191": "DISEASE",
    "T048": "DISEASE", "T049": "DISEASE", "T050": "DISEASE", "T190": "DISEASE",
    "T037": "DISEASE",
    # CHEMICAL_DRUG
    "T121": "CHEMICAL_DRUG", "T200": "CHEMICAL_DRUG", "T195": "CHEMICAL_DRUG",
    "T109": "CHEMICAL_DRUG", "T114": "CHEMICAL_DRUG", "T131": "CHEMICAL_DRUG",
    "T125": "CHEMICAL_DRUG", "T129": "CHEMICAL_DRUG", "T130": "CHEMICAL_DRUG",
    "T118": "CHEMICAL_DRUG", "T119": "CHEMICAL_DRUG", "T110": "CHEMICAL_DRUG",
    "T111": "CHEMICAL_DRUG", "T196": "CHEMICAL_DRUG", "T127": "CHEMICAL_DRUG",
    "T123": "CHEMICAL_DRUG", "T122": "CHEMICAL_DRUG", "T120": "CHEMICAL_DRUG",
    "T126": "CHEMICAL_DRUG", "T103": "CHEMICAL_DRUG", "T104": "CHEMICAL_DRUG",
    # PROCEDURE
    "T061": "PROCEDURE", "T060": "PROCEDURE", "T065": "PROCEDURE",
    "T058": "PROCEDURE", "T059": "PROCEDURE", "T063": "PROCEDURE",
    # ANATOMY
    "T023": "ANATOMY", "T024": "ANATOMY", "T017": "ANATOMY",
    "T029": "ANATOMY", "T030": "ANATOMY", "T025": "ANATOMY",
    "T018": "ANATOMY", "T021": "ANATOMY", "T022": "ANATOMY",
    "T026": "ANATOMY", "T031": "ANATOMY",
    # SIGN_SYMPTOM
    "T184": "SIGN_SYMPTOM", "T033": "SIGN_SYMPTOM", "T034": "SIGN_SYMPTOM",
    # ORGANISM
    "T001": "ORGANISM", "T002": "ORGANISM", "T004": "ORGANISM",
    "T005": "ORGANISM", "T007": "ORGANISM", "T008": "ORGANISM",
    "T010": "ORGANISM", "T011": "ORGANISM", "T012": "ORGANISM",
    "T013": "ORGANISM", "T014": "ORGANISM", "T015": "ORGANISM",
    "T016": "ORGANISM", "T096": "ORGANISM", "T101": "ORGANISM",
    # GENE_PROTEIN
    "T028": "GENE_PROTEIN", "T116": "GENE_PROTEIN",
    "T192": "GENE_PROTEIN", "T087": "GENE_PROTEIN", "T088": "GENE_PROTEIN",
    # MEDICAL_DEVICE
    "T074": "MEDICAL_DEVICE", "T075": "MEDICAL_DEVICE",
    "T203": "MEDICAL_DEVICE", "T073": "MEDICAL_DEVICE",
    # LAB_TEST
    "T201": "LAB_TEST",
}

NER_LABELS = sorted(set(UMLS_TYPE_TO_LABEL.values())) + ["OTHER"]
print("NER Labels:", NER_LABELS)


def get_label(ent_type: str) -> str:
    return UMLS_TYPE_TO_LABEL.get(ent_type, "OTHER")


def load_medmentions(split: str):
    """Load a MedMentions split → list of (text, {entities: [(s,e,label)]})."""
    print(f"Loading '{split}' split...")
    ds = load_dataset("Aremaki/MedMentions", name="Original", split=split)
    print(f"  {len(ds)} documents")

    data = []
    label_counts = Counter()
    skipped = 0

    for ex in ds:
        passages = ex.get("passages", [])
        entities = ex.get("entities", [])
        if not passages:
            continue

        full_text = ""
        for p in passages:
            for t in p.get("text", []):
                full_text += t + " "
        full_text = full_text.strip()
        if not full_text:
            continue

        doc_ents = []
        occupied = set()

        for ent in entities:
            label = get_label(ent.get("type", ""))
            for start, end in ent.get("offsets", []):
                start = max(0, min(start, len(full_text)))
                end = max(start, min(end, len(full_text)))
                if start == end:
                    continue
                char_range = set(range(start, end))
                if char_range & occupied:
                    skipped += 1
                    continue
                occupied.update(char_range)
                doc_ents.append((start, end, label))
                label_counts[label] += 1

        doc_ents.sort(key=lambda x: x[0])
        data.append((full_text, {"entities": doc_ents}))

    print(f"  {sum(label_counts.values())} entities ({skipped} overlaps skipped)")
    print(f"  Distribution: {dict(label_counts.most_common())}")
    return data


train_data = load_medmentions("train")
dev_data = load_medmentions("validation")
test_data = load_medmentions("test")

NER Labels: ['ANATOMY', 'CHEMICAL_DRUG', 'DISEASE', 'GENE_PROTEIN', 'LAB_TEST', 'MEDICAL_DEVICE', 'ORGANISM', 'PROCEDURE', 'SIGN_SYMPTOM', 'OTHER']
Loading 'train' split...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

original_data/train-00000-of-00001.parqu(…):   0%|          | 0.00/5.00M [00:00<?, ?B/s]

original_data/validation-00000-of-00001.(…):   0%|          | 0.00/1.68M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/2635 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/878 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/879 [00:00<?, ? examples/s]

  2635 documents
  122010 entities (231 overlaps skipped)
  Distribution: {'OTHER': 56177, 'CHEMICAL_DRUG': 22433, 'PROCEDURE': 14807, 'ANATOMY': 13652, 'SIGN_SYMPTOM': 9840, 'ORGANISM': 1827, 'MEDICAL_DEVICE': 1167, 'DISEASE': 1061, 'LAB_TEST': 1046}
Loading 'validation' split...
  878 documents


  40810 entities (74 overlaps skipped)
  Distribution: {'OTHER': 19000, 'CHEMICAL_DRUG': 7491, 'PROCEDURE': 4670, 'ANATOMY': 4473, 'SIGN_SYMPTOM': 3148, 'ORGANISM': 697, 'MEDICAL_DEVICE': 493, 'DISEASE': 434, 'LAB_TEST': 404}
Loading 'test' split...
  879 documents
  40083 entities (74 overlaps skipped)
  Distribution: {'OTHER': 18991, 'CHEMICAL_DRUG': 7400, 'PROCEDURE': 4776, 'ANATOMY': 4069, 'SIGN_SYMPTOM': 3197, 'ORGANISM': 621, 'MEDICAL_DEVICE': 355, 'DISEASE': 351, 'LAB_TEST': 323}


## 3. Convert to SpaCy DocBin

In [3]:
import spacy
from spacy.tokens import DocBin
from pathlib import Path

nlp = spacy.blank("en")


def to_docbin(data, output_path):
    db = DocBin()
    skipped = 0
    for text, ann in data:
        doc = nlp.make_doc(text)
        ents = []
        seen = set()
        for s, e, lbl in ann["entities"]:
            span = doc.char_span(s, e, label=lbl, alignment_mode="contract")
            if span is None:
                skipped += 1
                continue
            toks = set(range(span.start, span.end))
            if toks & seen:
                skipped += 1
                continue
            seen.update(toks)
            ents.append(span)
        doc.ents = ents
        db.add(doc)
    db.to_disk(output_path)
    total = sum(len(d[1]["entities"]) for d in data)
    print(f"  {output_path}: {len(data)} docs, {total - skipped} ents ({skipped} skipped)")


Path("data").mkdir(exist_ok=True)
to_docbin(train_data, "data/train.spacy")
to_docbin(dev_data, "data/dev.spacy")
to_docbin(test_data, "data/test.spacy")

  data/train.spacy: 2635 docs, 121860 ents (150 skipped)
  data/dev.spacy: 878 docs, 40766 ents (44 skipped)
  data/test.spacy: 879 docs, 40045 ents (38 skipped)


## 4. Training model

In [4]:
%%writefile config.cfg
[system]
gpu_allocator = null
seed = 42

[nlp]
lang = "en"
pipeline = ["tok2vec","ner"]
batch_size = 1000
disabled = []
before_creation = null
after_creation = null
after_pipeline_creation = null
tokenizer = {"@tokenizers":"spacy.Tokenizer.v1"}

[components]

[components.tok2vec]
factory = "tok2vec"

[components.tok2vec.model]
@architectures = "spacy.Tok2Vec.v2"

[components.tok2vec.model.embed]
@architectures = "spacy.MultiHashEmbed.v2"
width = 128
attrs = ["NORM","PREFIX","SUFFIX","SHAPE"]
rows = [5000,2500,2500,2500]
include_static_vectors = false

[components.tok2vec.model.encode]
@architectures = "spacy.MaxoutWindowEncoder.v2"
width = 128
depth = 4
window_size = 1
maxout_pieces = 3

[components.ner]
factory = "ner"
incorrect_spans_key = null
moves = null
scorer = {"@scorers":"spacy.ner_scorer.v1"}
update_with_oracle_cut_size = 100

[components.ner.model]
@architectures = "spacy.TransitionBasedParser.v2"
state_type = "ner"
extra_state_tokens = false
hidden_width = 64
maxout_pieces = 2
use_upper = true
nO = null

[components.ner.model.tok2vec]
@architectures = "spacy.Tok2VecListener.v1"
width = ${components.tok2vec.model.encode.width}
upstream = "*"

[paths]
train = null
dev = null
vectors = null
init_tok2vec = null

[training]
dev_corpus = "corpora.dev"
train_corpus = "corpora.train"
seed = ${system.seed}
gpu_allocator = ${system.gpu_allocator}
dropout = 0.1
accumulate_gradient = 1
patience = 1600
max_epochs = 0
max_steps = 20000
eval_frequency = 200
frozen_components = []
annotating_components = []
before_to_disk = null
before_update = null

[training.batcher]
@batchers = "spacy.batch_by_words.v1"
discard_oversize = false
tolerance = 0.2
get_length = null

[training.batcher.size]
@schedules = "compounding.v1"
start = 100
stop = 1000
compound = 1.001
t = 0.0

[training.logger]
@loggers = "spacy.ConsoleLogger.v1"
progress_bar = true

[training.optimizer]
@optimizers = "Adam.v1"
beta1 = 0.9
beta2 = 0.999
L2_is_weight_decay = true
L2 = 0.01
grad_clip = 1.0
use_averages = false
eps = 0.00000001

[training.optimizer.learn_rate]
@schedules = "warmup_linear.v1"
warmup_steps = 250
total_steps = 20000
initial_rate = 0.00005

[training.score_weights]
ents_per_type = null
ents_f = 1.0
ents_p = 0.0
ents_r = 0.0

[pretraining]

[corpora]

[corpora.dev]
@readers = "spacy.Corpus.v1"
path = ${paths.dev}
max_length = 0
gold_preproc = false
limit = 0
augmenter = null

[corpora.train]
@readers = "spacy.Corpus.v1"
path = ${paths.train}
max_length = 0
gold_preproc = false
limit = 0
augmenter = null

[initialize]
vectors = null
init_tok2vec = ${paths.init_tok2vec}
vocab_data = null
lookups = null
before_init = null
after_init = null

[initialize.components]

[initialize.tokenizer]

Writing config.cfg


In [5]:
!python -m spacy train config.cfg \
    --output ./models/custom_ner \
    --paths.train ./data/train.spacy \
    --paths.dev ./data/dev.spacy


✔ Created output directory: models/custom_ner
ℹ Saving to output directory: models/custom_ner
ℹ Using CPU

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.0
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00     64.32    0.00    0.00    0.00    0.00
  0     200         40.74  51073.24    0.00    0.00    0.00    0.00
  0     400        386.69  19875.28    0.00    0.00    0.00    0.00
  0     600        163.74  15044.96    0.00    0.00    0.00    0.00
  0     800        230.99  14421.90    0.16    9.62    0.08    0.00
  0    1000        362.94  14587.18    3.69   20.11    2.03    0.04
  0    1200        327.01  13320.26    5.92   21.34    3.44    0.06
  0    1400        339.21  13896.49    6.23 

## 5. Evaluate on Test Set

In [6]:
import spacy
from seqeval.metrics import classification_report, f1_score

# Load best model
nlp_trained = spacy.load("./models/custom_ner/model-best")
print(f"Pipeline: {nlp_trained.pipe_names}")
print(f"Labels: {nlp_trained.get_pipe('ner').labels}")


def evaluate(nlp_model, test_data):
    all_gold, all_pred = [], []

    for text, ann in test_data:
        doc = nlp_model(text)

        # Gold tags
        gold = ["O"] * len(doc)
        for s, e, lbl in ann["entities"]:
            span = doc.char_span(s, e, label=lbl, alignment_mode="contract")
            if span:
                for i in range(span.start, span.end):
                    gold[i] = f"{'B' if i == span.start else 'I'}-{lbl}"

        # Pred tags
        pred = ["O"] * len(doc)
        for ent in doc.ents:
            for i in range(ent.start, ent.end):
                pred[i] = f"{'B' if i == ent.start else 'I'}-{ent.label_}"

        all_gold.append(gold)
        all_pred.append(pred)

    print(classification_report(all_gold, all_pred, zero_division=0))
    print(f"Weighted F1: {f1_score(all_gold, all_pred, average='weighted', zero_division=0):.4f}")


evaluate(nlp_trained, test_data)

Pipeline: ['tok2vec', 'ner']
Labels: ('ANATOMY', 'CHEMICAL_DRUG', 'DISEASE', 'LAB_TEST', 'MEDICAL_DEVICE', 'ORGANISM', 'OTHER', 'PROCEDURE', 'SIGN_SYMPTOM')
                precision    recall  f1-score   support

       ANATOMY       0.46      0.21      0.29      4062
 CHEMICAL_DRUG       0.50      0.44      0.47      7384
       DISEASE       0.00      0.00      0.00       351
      LAB_TEST       0.91      0.10      0.17       323
MEDICAL_DEVICE       0.00      0.00      0.00       355
      ORGANISM       0.70      0.06      0.10       621
         OTHER       0.48      0.49      0.48     18981
     PROCEDURE       0.45      0.20      0.28      4773
  SIGN_SYMPTOM       0.57      0.10      0.17      3195

     micro avg       0.48      0.37      0.42     40045
     macro avg       0.45      0.18      0.22     40045
  weighted avg       0.48      0.37      0.40     40045

Weighted F1: 0.3952


## 6. Quick Test

In [7]:
text = (
    "The patient is a 65-year-old male presenting with severe dyspnea "
    "and hypertension. He was prescribed Lisinopril 20mg daily. "
    "Echocardiogram showed mild left ventricular hypertrophy."
)

doc = nlp_trained(text)
print(f"Text: {text}\n")
for ent in doc.ents:
    print(f"  {ent.text:30s}  {ent.label_}")

Text: The patient is a 65-year-old male presenting with severe dyspnea and hypertension. He was prescribed Lisinopril 20mg daily. Echocardiogram showed mild left ventricular hypertrophy.

  hypertension                    OTHER
  Echocardiogram                  OTHER
  left ventricular hypertrophy    OTHER
